# 训练系统里的数字从哪里来？

以一个线性层为起点，分别观察参数和激活的大小、一次乘法的工作量、张量的表示方式，以及反向传播的结果。各实验可在 CPU 上顺序执行。

## 从 token ID 到张量

两个长度为 4 的序列查同一张嵌入表，每个 ID 对应表中的一行。

In [1]:
import torch
from torch import nn

token_ids = torch.tensor(
    [
        [258, 259, 256, 257],
        [259, 258, 33, 257],
    ],
    dtype=torch.long,
)

print(token_ids.shape)
# torch.Size([2, 4])

V = 260  # 词表大小
d = 8    # 每个 token 用多少个浮点数表示

embedding = nn.Embedding(V, d, dtype=torch.float32)

h = embedding(token_ids)

print(embedding.weight.shape)
# torch.Size([260, 8])

print(h.shape)
# torch.Size([2, 4, 8])

print(torch.equal(h[0, 0], embedding.weight[258]))
# True

torch.Size([2, 4])
torch.Size([260, 8])
torch.Size([2, 4, 8])
True


## 线性层的参数、存储与 FLOPs

输入形状为 [B, T, d]，权重为 [d, m]。这里假设参数、梯度和两份 Adam 状态都是 FP32，只估算这四项占用。用一次乘法和一次加法合计 2 FLOPs 估算矩阵乘。反向同时计算输入和权重的梯度。

In [2]:
import torch


def linear_report(h, W):
    # 从形状中读取各个维度
    B, T, d = h.shape
    m = W.shape[1]

    # 执行前向矩阵乘法
    y = h @ W

    # 权重的元素总数，就是这层的参数量
    param_count = W.numel()

    # 一份权重的数据大小
    weight_bytes = W.numel() * W.element_size()

    # 参数、梯度、Adam 两份状态，共四份同样大小的数据
    param_state_bytes = 4 * weight_bytes

    # 计算输出 y 的数据大小
    output_bytes = y.numel() * y.element_size()

    # 根据尺寸估算前向 FLOPs
    forward_flops = 2 * B * T * d * m

    # 估算反向 FLOPs
    backward_flops = 2 * forward_flops

    return {
        "参数量": param_count,
        "参数相关四项存储（字节）": param_state_bytes,
        "输出存储（字节）": output_bytes,
        "前向 FLOPs": forward_flops,
        "反向 FLOPs": backward_flops,
    }


h = torch.randn(3, 5, 8, dtype=torch.float32, requires_grad=True)
W = torch.randn(8, 16, dtype=torch.float32, requires_grad=True)

print(linear_report(h, W))

{'参数量': 128, '参数相关四项存储（字节）': 2048, '输出存储（字节）': 960, '前向 FLOPs': 3840, '反向 FLOPs': 7680}


In [3]:
report = linear_report(h, W)
assert tuple(report.values()) == (128, 2048, 960, 3840, 7680)

## 算术强度和理想耗时

一次前向最少需要读取两个输入并写出结果。把运算量除以读写字节数，可以和设备的算力与带宽之比对照。示例的设备速率是便于手算的假定值。

In [4]:
# ** 表示乘方，例如 10 ** 9 就是 10 的 9 次方
flops = 10 ** 9
bytes_moved = 10 ** 9

compute_rate = 10 ** 12       # FLOP/s
memory_bandwidth = 10 ** 11   # 字节/s

compute_time = flops / compute_rate
memory_time = bytes_moved / memory_bandwidth

# max 取两个时间中较大的那个
ideal_time = max(compute_time, memory_time)

print(compute_time * 1000)  # 1.0 ms
print(memory_time * 1000)   # 10.0 ms
print(ideal_time * 1000)    # 10.0 ms

1.0
10.0
10.0


In [5]:
flops = 10 ** 9
bytes_moved = 10 ** 9

compute_rate = 10 ** 12
memory_bandwidth = 10 ** 11

arithmetic_intensity = flops / bytes_moved
hardware_ratio = compute_rate / memory_bandwidth

print(arithmetic_intensity)  # 1.0 FLOP/字节
print(hardware_ratio)        # 10.0 FLOP/字节

1.0
10.0


In [6]:
B, T, d = h.shape
m = W.shape[1]
y = h @ W
weight_bytes = W.numel() * W.element_size()
output_bytes = y.numel() * y.element_size()
forward_flops = 2 * B * T * d * m

# 输入 h 的数据大小
input_bytes = h.numel() * h.element_size()

# 理想情况下，两个输入各读一次，输出写一次
io_bytes = input_bytes + weight_bytes + output_bytes

# 每读写一个字节，完成多少次浮点运算
arithmetic_intensity = forward_flops / io_bytes
print("线性层的算术强度", arithmetic_intensity, "FLOP/字节")

线性层的算术强度 1.9672131147540983 FLOP/字节


### 实测 CPU 矩阵乘

先预热，再测重复前向的平均耗时。输出取决于当前设备与 PyTorch 线程设置。

In [7]:
import time
import torch

# 明确指定在 CPU 上运行，只测前向矩阵乘法
h = torch.randn(1, 64, 128, dtype=torch.float32, device="cpu")
W = torch.randn(128, 128, dtype=torch.float32, device="cpu")

# 先运行几次，减少首次初始化对计时的影响
for _ in range(20):
    y = h @ W

# 重复执行并计时
repeats = 1000

start = time.perf_counter()

for _ in range(repeats):
    y = h @ W

elapsed = time.perf_counter() - start

# 总秒数除以执行次数，得到平均每次的秒数
avg_seconds = elapsed / repeats

# 计算一次前向的近似 FLOPs
B, T, d = h.shape
m = W.shape[1]
forward_flops = 2 * B * T * d * m

# 工作量除以实测时间，得到有效计算速度
effective_rate = forward_flops / avg_seconds

print("平均前向耗时（毫秒）：", avg_seconds * 1000)
print("有效计算速度（FLOP/s）：", effective_rate)

平均前向耗时（毫秒）： 0.04837729999417206
有效计算速度（FLOP/s）： 43349918252.00333


## 数据类型、设备与混合精度

观察三个数量级差别很大的数在 FP32、FP16 和 BF16 中的表示；矩阵乘用 CPU 的 BF16 自动混合精度执行。

In [8]:
import torch

x = torch.randn(
    3, 5, 8,
    dtype=torch.float32,
    device="cpu"
)

print(x.shape)   # torch.Size([3, 5, 8])
print(x.dtype)   # torch.float32
print(x.device)  # cpu

values = torch.tensor(
    [1.001, 1e-8, 100000.0],
    dtype=torch.float32
)

for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    converted = values.to(dtype=dtype)

    print("类型：", converted.dtype)
    print("每个元素的字节数：", converted.element_size())
    print("数值：", converted.tolist())
    print()

x16 = x.to(dtype=torch.float16)

print(x.dtype)    # torch.float32
print(x16.dtype)  # torch.float16

torch.Size([3, 5, 8])
torch.float32
cpu
类型： torch.float32
每个元素的字节数： 4
数值： [1.0010000467300415, 9.99999993922529e-09, 100000.0]

类型： torch.float16
每个元素的字节数： 2
数值： [1.0009765625, 0.0, inf]

类型： torch.bfloat16
每个元素的字节数： 2
数值： [1.0, 1.0011717677116394e-08, 99840.0]

torch.float32
torch.float16


In [9]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print("本次使用的设备：", device)

h = torch.randn(
    3, 5, 8,
    dtype=torch.float32,
    device=device
)

W = torch.randn(
    8, 16,
    dtype=torch.float32,
    device=device
)

y = h @ W

print("h：", h.shape, h.dtype, h.device)
print("W：", W.shape, W.dtype, W.device)
print("y：", y.shape, y.dtype, y.device)

x_cpu = torch.randn(3, 5, 8, device="cpu")

# 将返回的张量保存在目标设备上
x_on_device = x_cpu.to(device=device)

# 也可以同时指定设备和类型
x_on_device = x_cpu.to(
    device=device,
    dtype=torch.float32
)

token_ids = torch.tensor(
    [[258, 259, 256, 257]],
    dtype=torch.long,
    device="cpu"
)

token_ids = token_ids.to(device=device)

print(token_ids.dtype)   # torch.int64
print(token_ids.device)

本次使用的设备： cpu
h： torch.Size([3, 5, 8]) torch.float32 cpu
W： torch.Size([8, 16]) torch.float32 cpu
y： torch.Size([3, 5, 16]) torch.float32 cpu
torch.int64
cpu


In [10]:
a = torch.randn(3, 5, 8, dtype=torch.float32, device="cpu")
b = torch.randn(8, 16, dtype=torch.float32, device="cpu")

with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    c = a @ b

print("a 的类型：", a.dtype)
print("b 的类型：", b.dtype)
print("c 的类型：", c.dtype)

a 的类型： torch.float32
b 的类型： torch.float32
c 的类型： torch.bfloat16


## 形状、内存布局与命名维度

从同一组 24 个数出发，比较改变形状、交换维度和让内存连续这几种操作，再用 einops 的维度表达式复现。

In [11]:
import torch

x = torch.arange(
    24,
    dtype=torch.float32
).reshape(2, 3, 4)

B, T, d = x.shape

print(x)

tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])


In [12]:
print(x[0])        # 第一条样本，形状 [3, 4]
print(x[0, 1])     # 第一条样本的第二个 token，形状 [4]
print(x[0, 1, 2])  # 这个 token 的第三个特征，数值 6

flat = x.reshape(B * T, d)

print(flat.shape)  # torch.Size([6, 4])
print(flat)

restored = flat.reshape(B, T, d)

print(torch.equal(restored, x))  # True

transposed = x.transpose(1, 2)

print(x.shape)           # [2, 3, 4]
print(transposed.shape)  # [2, 4, 3]

transposed = x.transpose(1, 2)
reshaped = x.reshape(2, 4, 3)

print(transposed.shape)  # [2, 4, 3]
print(reshaped.shape)   # [2, 4, 3]

print(transposed[0, 0])  # tensor([0., 4., 8.])
print(reshaped[0, 0])    # tensor([0., 1., 2.])

permuted = x.permute(0, 2, 1)

print(torch.equal(permuted, transposed))  # True

print(x.is_contiguous())           # True
print(transposed.is_contiguous())  # False

flat_view = x.view(6, 4)

packed = transposed.contiguous()
flat_transposed = packed.view(2, 12)

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
tensor([4., 5., 6., 7.])
tensor(6.)
torch.Size([6, 4])
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.],
        [20., 21., 22., 23.]])
True
torch.Size([2, 3, 4])
torch.Size([2, 4, 3])
torch.Size([2, 4, 3])
torch.Size([2, 4, 3])
tensor([0., 4., 8.])
tensor([0., 1., 2.])
True
True
False


In [13]:
from einops import rearrange, reduce, einsum

flat_named = rearrange(x, "b t d -> (b t) d")
swapped_named = rearrange(x, "b t d -> b d t")

print(flat_named.shape)     # [6, 4]
print(swapped_named.shape)  # [2, 4, 3]

restored_named = rearrange(
    flat_named,
    "(b t) d -> b t d",
    b=B,
    t=T
)

print(torch.equal(restored_named, x))  # True

grouped = rearrange(
    x,
    "b t (heads dh) -> b heads t dh",
    heads=2
)

print(grouped.shape)  # [2, 2, 3, 2]

grouped_torch = x.reshape(B, T, 2, 2).permute(0, 2, 1, 3)

average = x.mean(dim=1)

print(average.shape)  # [2, 4]
print(average)

average_named = reduce(
    x,
    "b t d -> b d",
    "mean"
)

print(torch.equal(average_named, average))  # True

W = torch.randn(4, 6, dtype=torch.float32)

y_matmul = x @ W

y_einsum = einsum(
    x,
    W,
    "b t d, d m -> b t m"
)

print(y_einsum.shape)  # [2, 3, 6]
print(torch.allclose(y_matmul, y_einsum))  # True

torch.Size([6, 4])
torch.Size([2, 4, 3])
True
torch.Size([2, 2, 3, 2])
torch.Size([2, 4])
tensor([[ 4.,  5.,  6.,  7.],
        [16., 17., 18., 19.]])
True
torch.Size([2, 3, 6])
True


In [14]:
q_raw = torch.randn(2, 3, 8)
k_raw = torch.randn(2, 3, 8)

q = rearrange(
    q_raw,
    "b t (heads dh) -> b heads t dh",
    heads=2
)

k = rearrange(
    k_raw,
    "b t (heads dh) -> b heads t dh",
    heads=2
)

print(q.shape)  # [2, 2, 3, 4]
print(k.shape)  # [2, 2, 3, 4]

scores = einsum(
    q,
    k,
    "b heads i dh, b heads j dh -> b heads i j"
)

print(scores.shape)  # [2, 2, 3, 3]

scores_matmul = q @ k.transpose(-2, -1)

print(torch.allclose(scores, scores_matmul))  # True

torch.Size([2, 2, 3, 4])
torch.Size([2, 2, 3, 4])
torch.Size([2, 2, 3, 3])
True


## 整批更新与梯度累积

8 条样本先整体做一次 SGD 更新，再分成 4 个等大的微批次，各自缩放损失、累积梯度，最后只做一次更新。两边从同一初始权重开始。

In [15]:
import torch

torch.manual_seed(0)

# 8 条样本，每条 4 个输入特征
X = torch.randn(8, 4, dtype=torch.float32)

# 每条样本对应一个目标值
target = torch.randn(8, 1, dtype=torch.float32)

# 保存一份初始权重，便于后面公平对比
W0 = torch.randn(4, 1, dtype=torch.float32)

# 创建真正参与本次训练的权重
W_full = W0.clone().requires_grad_(True)

# 告诉优化器：它需要更新 W_full
opt_full = torch.optim.SGD([W_full], lr=0.1)

# 1. 清理之前的梯度
opt_full.zero_grad(set_to_none=True)

# 2. 前向：得到预测值
pred_full = X @ W_full

# 3. 计算一个标量损失：均方误差
loss_full = ((pred_full - target) ** 2).mean()

# 4. 反向：计算损失对 W_full 的梯度
loss_full.backward()

# 留下梯度副本，稍后用于比较
grad_full = W_full.grad.clone()

# 5. 根据梯度更新权重
opt_full.step()

In [16]:
# 从相同的初始权重开始
W_acc = W0.clone().requires_grad_(True)

opt_acc = torch.optim.SGD([W_acc], lr=0.1)

micro_batch_size = 2
accum_steps = 4

# 一整组累积开始前，只清理一次
opt_acc.zero_grad(set_to_none=True)

for start in range(0, 8, micro_batch_size):
    # 每次取出 2 条样本
    x_micro = X[start:start + micro_batch_size]
    target_micro = target[start:start + micro_batch_size]

    # 当前小批次的前向计算
    pred_micro = x_micro @ W_acc

    # 当前小批次内部的平均损失
    loss_micro = ((pred_micro - target_micro) ** 2).mean()

    # 调整为它在整个批次平均损失中的贡献
    loss_for_backward = loss_micro / accum_steps

    # 梯度自动累加到 W_acc.grad
    loss_for_backward.backward()

# 四个小批次全部完成后，保存累计梯度
grad_acc = W_acc.grad.clone()

# 只更新一次参数
opt_acc.step()

print(
    "梯度是否接近：",
    torch.allclose(grad_full, grad_acc, atol=1e-6)
)

print(
    "更新后的参数是否接近：",
    torch.allclose(W_full, W_acc, atol=1e-6)
)
assert torch.allclose(grad_full, grad_acc, atol=1e-6)
assert torch.allclose(W_full, W_acc, atol=1e-6)

梯度是否接近： True
更新后的参数是否接近： True


## 激活重计算

同一个双层网络保留两份相同的初始参数。一份直接反传，另一份在反向时重算中间激活，对照输出和两层权重梯度。

In [17]:
from torch.utils.checkpoint import checkpoint

def block(x, W1, W2):
    hidden = torch.relu(x @ W1)
    return hidden @ W2


# 两种计算方式共用相同的初始数值
W1_initial = torch.randn(4, 16)
W2_initial = torch.randn(16, 1)

# 普通版本的参数
W1_plain = W1_initial.clone().requires_grad_(True)
W2_plain = W2_initial.clone().requires_grad_(True)

# 重计算版本的参数
W1_ckpt = W1_initial.clone().requires_grad_(True)
W2_ckpt = W2_initial.clone().requires_grad_(True)


# ---------- 普通版本 ----------

out_plain = block(X, W1_plain, W2_plain)

loss_plain = ((out_plain - target) ** 2).mean()
loss_plain.backward()


# ---------- 激活重计算版本 ----------

out_ckpt = checkpoint(
    block,
    X,
    W1_ckpt,
    W2_ckpt,
    use_reentrant=False
)

loss_ckpt = ((out_ckpt - target) ** 2).mean()
loss_ckpt.backward()


# ---------- 对比结果 ----------

print(
    "输出是否接近：",
    torch.allclose(out_plain, out_ckpt, atol=1e-6)
)

print(
    "W1 梯度是否接近：",
    torch.allclose(W1_plain.grad, W1_ckpt.grad, atol=1e-6)
)

print(
    "W2 梯度是否接近：",
    torch.allclose(W2_plain.grad, W2_ckpt.grad, atol=1e-6)
)
assert torch.allclose(out_plain, out_ckpt, atol=1e-6)
assert torch.allclose(W1_plain.grad, W1_ckpt.grad, atol=1e-6)
assert torch.allclose(W2_plain.grad, W2_ckpt.grad, atol=1e-6)

输出是否接近： True
W1 梯度是否接近： True
W2 梯度是否接近： True
